# Colab Setup
Start by cloning the PyCCAPT repository into the Colab session. Re-run this section whenever the runtime is reset.


In [ ]:
from pathlib import Path

repo_dir = Path('/content/pyccapt')
if repo_dir.exists():
    print(f'Using existing repository at {repo_dir}')
else:
    !git clone --branch main https://github.com/mmonajem/pyccapt.git /content/pyccapt


Install the notebook dependencies from the package extras instead of listing them one by one. This keeps the Colab setup aligned with the repository metadata.


In [ ]:
print('Dependencies will be installed from the repository extras after we enter /content/pyccapt.')


Move into the cloned repository before installing the package.


In [ ]:
%cd /content/pyccapt
%ls


Install PyCCAPT with the `calibration` extra so the tutorial dependencies come from the package definition. `gdown` is included here because the example data-download cells use it.


In [ ]:
%%capture
%pip install -U pip gdown
%pip install -e ".[calibration]"


# Data Processing Workflow
Use this notebook to load a dataset, crop it, calibrate the mass spectrum, reconstruct the volume, and save the final outputs. Run the cells from top to bottom the first time through, then return to the calibration sections as needed.


In [ ]:
# Enable the custom widget manager for Colab and choose the best matplotlib backend available.
from google.colab import output
from IPython import get_ipython

output.enable_custom_widget_manager()

def _activate_matplotlib_backend():
    shell = get_ipython()
    for backend in ("widget", "notebook"):
        try:
            shell.run_line_magic("matplotlib", backend)
            print(f"Using matplotlib backend: {backend}")
            return backend
        except Exception as exc:
            print(f"Could not enable %matplotlib {backend}: {exc}")
    shell.run_line_magic("matplotlib", "inline")
    print("Falling back to %matplotlib inline. Plots still work, but interactive drawing tools may be limited in this runtime. Use the typed crop/index fields if needed.")
    return "inline"

COLAB_MATPLOTLIB_BACKEND = _activate_matplotlib_backend()
# Activate auto reload
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
# import libraries
import numpy as np
from IPython.display import HTML, clear_output, display
import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

# Local module and scripts
from pyccapt.calibration.core import widgets as wd
from pyccapt.calibration.data_tools import data_tools, data_loadcrop
from pyccapt.calibration.tutorials.tutorials_helpers import helper_calibration
from pyccapt.calibration.tutorials.tutorials_helpers import helper_data_loader
from pyccapt.calibration.tutorials.tutorials_helpers import helper_temporal_crop
from pyccapt.calibration.tutorials.tutorials_helpers import helper_special_crop
from pyccapt.calibration.tutorials.tutorials_helpers import helper_t_0_tune
from pyccapt.calibration.tutorials.tutorials_helpers import helper_mc_plot
from pyccapt.calibration.tutorials.tutorials_helpers import helper_3d_reconstruction
from pyccapt.calibration.tutorials.tutorials_helpers import helper_ion_selection
from pyccapt.calibration.tutorials.tutorials_helpers import helper_visualization
from pyccapt.calibration.tutorials.tutorials_helpers import helper_ion_list
from pyccapt.calibration.core import ion_selection
from pyccapt.calibration.core import share_variables


If `pytables` is missing on your Colab runtime, install it in a new notebook cell before continuing:

`%pip install tables`


Create the shared `variables` object first. Most helper functions read from and write to this object so the state stays synchronized across the workflow.


In [ ]:
# Create the shared state container used throughout the tutorial.
variables = share_variables.Variables()

Download the example dataset or replace the Google Drive file ID with your own dataset. For large files, Google Drive is much faster than a direct upload in Colab.


In [ ]:
! gdown https://drive.google.com/uc?id=1JveMf6m1BxCAUaum-_ZVn1_WCN1SynAy
dataset_path = '/content/pyccapt/2382_Jan-10-2025_15-12_NiC9_Al_cropped.h5'

In [ ]:
# Uncomment the lines below to upload your own dataset instead of using the example file.
# from google.colab import files
# uploaded = files.upload()
# dataset_path = next(iter(uploaded))


## ROI Selection And Data Cropping
This section loads the data, lets you inspect the experiment history, and trims the dataset before calibration.


Set the instrument metadata for the dataset before loading it. These values control how time-of-flight, mass-to-charge, and detector geometry are interpreted in the following cells.


In [ ]:
# Create an object for selection of instrument specifications of the dataset
tdc, pulse_mode, flight_path_length, t0, max_mc, det_diam = wd.dataset_instrument_specification_selection()

# Display lists and comboboxes to select instrument specifications
display(tdc, pulse_mode, flight_path_length, t0, det_diam, max_mc)

In [ ]:
# Load the dataset with the selected instrument settings and preview the loaded tables.
helper_data_loader.load_data(dataset_path, max_mc.value, flight_path_length.value, pulse_mode.value, tdc.value, variables)
data_tools.extract_data(variables.data, variables, flight_path_length.value, max_mc.value)
display(variables.data)
display(variables.range_data)

If you already have a saved range file (`.h5`, `.rrng`, or `.rng`), load it here so you can reuse previous ion labels instead of rebuilding the range table from scratch.


In [ ]:
# Optional example range file download. Uncomment these lines if you want to preload a saved range table.
# ! gdown <google-drive-file-id>
# range_path = '/content/pyccapt/example_range.h5'


In [ ]:
# Uncomment the lines below to upload your own saved range file.
# from google.colab import files
# uploaded = files.upload()
# range_path = next(iter(uploaded))


In [ ]:
# If a range file was chosen, load it into the shared state and preview it.
if 'range_path' in globals():
    variables.range_data = data_tools.read_range(range_path)
display(variables.range_data)

# Temporal Crop
Draw a rectangle on the experiment-history plot to keep the useful evaporation interval and remove unstable regions.


In [ ]:
helper_temporal_crop.call_plot_crop_experiment(variables, pulse_mode.value)

# Spatial Crop
Draw a region on the field desorption map to keep the detector area with the strongest, cleanest signal.


In [ ]:
helper_special_crop.call_plot_crop_fdm(variables)

The next cell recomputes pulse statistics after cropping and prints quick quality checks, including ROI loss and the multihit fraction.


In [ ]:
# maximum value of the tdc counter
if tdc.value == 'pyccapt':
    if not (variables.data['delta_p'] != 0).any() or not (variables.data['multi'] != 0).any():
        if (variables.data['start_counter'] != 0).any():
            max_start_counter = max(variables.data['start_counter'])
            pulse_pi, ion_pp = data_loadcrop.calculate_ppi_and_ipp(variables.data, max_start_counter)
            variables.data['delta_p'] = pulse_pi.astype(np.uintc)
            variables.data['multi'] = ion_pp.astype(np.uintc)

    print('tof Crop Loss {:.2f} %'.format((100 - (len(variables.data) / len(variables.data_backup)) * 100)))
# percentage of multihit event per pulse
ion_pp = variables.data['multi'].to_numpy()
print('percentage of multihit event per pulse', len(ion_pp[ion_pp != 1]) / float(len(ion_pp)) * 100)

Before mass calibration, refresh the extracted arrays and optionally fine-tune `t_0` so a known peak lands at the expected position.


In [ ]:
# Refresh the extracted arrays before opening the interactive t0 tuning helper.
data_tools.extract_data(variables.data, variables, flight_path_length.value, max_mc.value)
variables.data

In [ ]:
# Launch the fine-tuning widget for t0 using the current dataset and settings.
helper_t_0_tune.call_fine_tune_t_0(variables, flight_path_length, pulse_mode, t0)

This step removes invalid rows, adds the columns needed for later calibration and reconstruction, and writes a temporary checkpoint to disk.


In [ ]:
# Add calibration-ready columns, save a temporary snapshot, and inspect the updated schema.
helper_data_loader.add_columns(variables, max_mc)
data_tools.save_data(variables.data, variables, hdf=True, epos=False, pos=False, ato_6v=False, csv=False, temp=True)
display(variables.data)
display(variables.data.dtypes)

---

## Time-Of-Flight And Mass-To-Charge Calibration
Use this section iteratively: inspect the histogram, select peaks, apply corrections, and repeat until the peak shapes stop improving.

Start by refreshing the extracted arrays, then open the calibration helper. After each adjustment, save the updated calibrated values back into `variables.data`.

In [ ]:
# Refresh the extracted arrays before running the voltage and bowl calibration helper.
data_tools.extract_data(variables.data, variables, flight_path_length.value, max_mc.value)

In [ ]:
# Open the interactive voltage and bowl calibration workflow.
helper_calibration.call_voltage_bowl_calibration(variables, det_diam.value, flight_path_length.value, pulse_mode.value, t0.value)

In [ ]:
# Keep a backup copy of the current calibrated arrays before further edits.
variables.dld_t_calib_backup = np.copy(variables.dld_t_calib)
variables.mc_calib_backup = np.copy(variables.mc_calib)

In [ ]:
helper_ion_list.call_ion_list(variables, selector='peak', mode='tof')

**Optional: Absolute m/c Position Correction**

If the calibrated peaks are slightly offset from their true m/c positions, use the cell below to shift them onto the correct values.
Two methods are available:

- **NIST auto-match** — automatically detects peaks and matches them to the expected m/c values you added via the ADD button (no peak clicking needed).
- **Parametric fit** — lets you manually click each peak in the plot and fit a polynomial correction.

Both methods only nudge the m/c axis to the absolute reference and are safe to run on top of any prior calibration.

In [ ]:
helper_ion_list.call_ion_list(variables, selector='peak', mode='mc')

In [ ]:
# Write the calibrated values back to the dataframe, drop invalid negative masses, and save a checkpoint.
variables.data['mc (Da)'] = variables.mc_calib
variables.data['t_c (ns)'] = variables.dld_t_calib
# Remove negative mc
threshold = 0
mc_t = variables.data['mc (Da)'].to_numpy()
mc_t_mask = (mc_t <= threshold)
print('The number of ions with negative mc are:', len(mc_t_mask[mc_t_mask==True]))
variables.data.drop(np.where(mc_t_mask)[0], inplace=True)
variables.data.reset_index(inplace=True, drop=True)
data_tools.save_data(variables.data, variables, hdf=True, epos=False, pos=False, ato_6v=False, csv=False, temp=True)

In [ ]:
helper_mc_plot.call_mc_plot(variables, selector='None')

---

## 3D Reconstruction
Once the calibration looks stable, compute reconstructed `x`, `y`, and `z` coordinates and save them back into the working dataset.


In [ ]:
# Refresh the extracted arrays before reconstruction and inspect the current dataframe.
data_tools.extract_data(variables.data, variables, flight_path_length.value, max_mc.value)

display(variables.data)

Choose the matrix or dominant element in the sample before running the reconstruction helper. That value is used during density-field reconstruction.


In [ ]:
# Display the selector used to choose the matrix element for reconstruction.
element_selected = wd.density_field_selection()
display(element_selected)

If Plotly fails to render in Colab, restart the runtime and rerun the setup cells so the required frontend assets are registered again.


In [ ]:
# Compute the reconstructed x, y, and z coordinates.
helper_3d_reconstruction.call_x_y_z_calculation(variables, flight_path_length, element_selected=element_selected, colab=True)

In [ ]:
# Store the reconstructed coordinates in the dataframe and save a temporary copy.
variables.data['x (nm)'] = variables.x
variables.data['y (nm)'] = variables.y
variables.data['z (nm)'] = variables.z
data_tools.save_data(variables.data, variables, hdf=True, epos=False, pos=False, ato_6v=False, csv=False, temp=True)

---

## Ion Selection And Ranging
In this section you assign peaks to ions, review the range table, and save both the range definition and the processed dataset.


In [ ]:
# Refresh the extracted arrays before opening the ranging interface.
data_tools.extract_data(variables.data, variables, flight_path_length.value, max_mc.value)

display(variables.data)

In [ ]:
display(variables.data.dtypes)

In [ ]:
# Open the ion-selection and ranging helper.
helper_ion_selection.call_ion_selection(variables, colab=True)

The styled table below lets you verify the range colors before saving the range file.

In [ ]:
# Preview the range table with its display colors applied.
display(variables.range_data.style.applymap(ion_selection.display_color, subset=['color']))

In [ ]:
variables.range_data.dtypes

Use the next cell to export the current range table once the ion list and colors look correct.


In [ ]:
# Export the current range table from Colab.
from google.colab import files
variables.range_data.to_hdf('range_' + variables.dataset_name + '.h5', key='df', mode='w')
variables.range_data.to_csv('range_' + variables.dataset_name + '.csv', encoding='utf-8', index=False, sep=';')
files.download('range_' + variables.dataset_name + '.h5')
files.download('range_' + variables.dataset_name + '.csv')

Use the next cell to export the processed dataset in the formats you need.


In [ ]:
# Export the calibrated dataset in the formats you need for downstream work.
# By default the whole dataset is exported; change the indices below if you want a smaller subset.
export_name = f'{variables.dataset_name}_calibrated'
export_last_index = max(0, len(variables.data) - 1)
export_start_index = 0
export_end_index = export_last_index
data_tools.save_data(
    variables.data,
    variables,
    name=export_name,
    hdf=True,
    epos=False,
    pos=False,
    ato_6v=True,
    csv=False,
    start_index=export_start_index,
    end_index=export_end_index,
)
print(f'HDF5 export: {variables.resolve_result_data_file(export_name + ".h5")}')
print(f'ATO export: {variables.resolve_result_file(export_name + ".ato")}')
# data_tools.save_data(variables.data, variables, name=export_name, hdf=False, epos=True, pos=False, ato_6v=False, csv=False, start_index=export_start_index, end_index=export_end_index)
# data_tools.save_data(variables.data, variables, name=export_name, hdf=False, epos=False, pos=True, ato_6v=False, csv=False, start_index=export_start_index, end_index=export_end_index)


---

## Visualization
Use the final cells to inspect the reconstructed dataset interactively and open the main visualization helper.


For large datasets, use external HTML rendering by default to keep the notebook responsive. You can switch to inline mode when needed.


In [ ]:
# Display/performance defaults. In Colab, inline shows directly in the notebook; external saves a downloadable HTML.
VIS_DISPLAY_MODE = 'external'  # Options: 'external', 'inline', 'skip'
SHOW_DETECTOR_ANIMATION = False
CLEAR_PREVIOUS_VIS_OUTPUT = True


In [ ]:
variables.data

In [ ]:
# Refresh the extracted arrays before launching the visualization helper.
data_tools.extract_data(variables.data, variables, flight_path_length.value, max_mc.value)

In [ ]:
# Open the Colab-compatible visualization helper.
# Because Colab does not support ipywidgets.Tab the same way as local Jupyter, use the dedicated Colab mode.
helper_visualization.call_visualization(variables, colab=True)

In [ ]:
if CLEAR_PREVIOUS_VIS_OUTPUT:
    # Keep only the latest heavy figure output visible in this cell.
    clear_output(wait=True)

plotly_fig = getattr(variables, 'plotly_3d_reconstruction', None)
if plotly_fig is None:
    print('No reconstruction figure is available yet. Run reconstruction and visualization first.')
elif VIS_DISPLAY_MODE == 'inline':
    plotly_fig.show()
elif VIS_DISPLAY_MODE == 'external':
    html_path = variables.resolve_result_file('reconstruction_view.html')
    plotly_fig.write_html(html_path, include_plotlyjs='cdn')
    print(f'Reconstruction HTML saved to: {html_path}')
    print('Open this file in your browser for smoother interaction.')
else:
    print('Skipping reconstruction figure display (VIS_DISPLAY_MODE="skip").')


In [ ]:
if SHOW_DETECTOR_ANIMATION and getattr(variables, 'animation_detector_html', None):
    display(HTML(variables.animation_detector_html))
else:
    print('Detector animation is disabled by default to keep notebook rendering fast.')